# Modeling / Architecture

This phase builds the first collaborative recommendation model.

## Objective

Use historical user-item ratings to learn personalized product
preference scores, which will later be used to rank candidate products.

## Model

Explicit Matrix Factorization implemented in PyTorch.

Ratings are centered around 3 stars:

- 1 → -1.0
- 2 → -0.5
- 3 → 0.0
- 4 → +0.5
- 5 → +1.0

The model learns separate latent representations for users and products.

## Important cold-start limitation

The collaborative model only learns item representations for products
with interactions in the training period (≤ 2021).

Products with zero training interactions remain in the global catalog
but cannot receive a collaborative score. This limitation is intentional
and will later be addressed using content-based representations.

In [4]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd().parent

INTERACTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "electronics_interactions"
    / "*.parquet"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

con = duckdb.connect()

In [5]:
users = con.sql(f"""
    SELECT DISTINCT user_id
    FROM read_parquet('{INTERACTION_PATH}')
    WHERE timestamp < 1640995200000
    ORDER BY user_id
""").df()

users["user_idx"] = np.arange(
    len(users),
    dtype=np.int32,
)

users = users[["user_idx", "user_id"]]

print(f"Training users: {len(users):,}")
users.head()

Training users: 16,058,864


,user_idx,user_id
0,0,AE2222C2STTMXNUHXTVDF55R7E3Q
1,1,AE2222FRPDMNOMYOMCWIANTXP7UQ
2,2,AE2222VGHFD37G7NOCWXASYY3ZDQ
3,3,AE2222WEGI5MUXLFZW6UKKUSI74A
4,4,AE22232OB6S7UAC75JUFYRGBSHIQ


In [6]:
mf_items = con.sql(f"""
    SELECT DISTINCT parent_asin
    FROM read_parquet('{INTERACTION_PATH}')
    WHERE timestamp < 1640995200000
    ORDER BY parent_asin
""").df()

mf_items["mf_item_idx"] = np.arange(
    len(mf_items),
    dtype=np.int32,
)

mf_items = mf_items[["mf_item_idx", "parent_asin"]]

print(f"Training items: {len(mf_items):,}")
mf_items.head()

Training items: 1,395,130


,mf_item_idx,parent_asin
0,0,0007182147
1,1,0011300000
2,2,0060009810
3,3,0060219602
4,4,0060897082


In [5]:
users.to_parquet(
    OUTPUT_DIR / "mf_users.parquet",
    index=False,
)

mf_items.to_parquet(
    OUTPUT_DIR / "mf_items.parquet",
    index=False,
)

In [7]:
assert users["user_idx"].is_unique
assert users["user_id"].is_unique

assert mf_items["mf_item_idx"].is_unique
assert mf_items["parent_asin"].is_unique

assert users["user_idx"].iloc[0] == 0
assert mf_items["mf_item_idx"].iloc[0] == 0

print("MF index mappings validated.")

MF index mappings validated.


In [8]:
rating_to_target = {
    1.0: -1.0,
    2.0: -0.5,
    3.0: 0.0,
    4.0: 0.5,
    5.0: 1.0,
}

In [9]:
train_interactions = con.sql(f"""
    SELECT
        u.user_idx,
        i.mf_item_idx,
        CASE rating
            WHEN 1.0 THEN -1.0
            WHEN 2.0 THEN -0.5
            WHEN 3.0 THEN  0.0
            WHEN 4.0 THEN  0.5
            WHEN 5.0 THEN  1.0
        END AS target
    FROM read_parquet('{INTERACTION_PATH}') r
    INNER JOIN users u
        ON r.user_id = u.user_id
    INNER JOIN mf_items i
        ON r.parent_asin = i.parent_asin
    WHERE EXTRACT(
        YEAR FROM to_timestamp(r.timestamp / 1000)
    ) <= 2021
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
print(train_interactions.shape)
print(train_interactions.head())
print(train_interactions.dtypes)
print(train_interactions["target"].value_counts().sort_index())

(37006469, 3)
   user_idx  mf_item_idx  target
0    102400      1388149     1.0
1    102401       618441     1.0
2    102402       883214     0.5
3    102403       999968    -1.0
4    102404      1222463     1.0
user_idx         int32
mf_item_idx      int32
target         float64
dtype: object
target
-1.0     4285643
-0.5     1864074
 0.0     2419023
 0.5     4840173
 1.0    23597556
Name: count, dtype: int64


### Preparing data

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

user_indices = train_interactions["user_idx"].to_numpy(dtype=np.int64)
item_indices = train_interactions["mf_item_idx"].to_numpy(dtype=np.int64)
targets = train_interactions["target"].to_numpy(dtype=np.float32)

print(user_indices.nbytes / 1024**2, "MB")
print(item_indices.nbytes / 1024**2, "MB")
print(targets.nbytes / 1024**2, "MB")

282.33695220947266 MB
282.33695220947266 MB
141.16847610473633 MB


### MF Dataset

In [12]:
class MFDataset(Dataset):
    def __init__(self, user_indices, item_indices, targets):
        self.user_indices = user_indices
        self.item_indices = item_indices
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.user_indices[idx],
            self.item_indices[idx],
            self.targets[idx],
        )

In [13]:
train_dataset = MFDataset(
    user_indices,
    item_indices,
    targets,
)

print("Training examples:", len(train_dataset))
print("First example:", train_dataset[0])

Training examples: 37006469
First example: (np.int64(102400), np.int64(1388149), np.float32(1.0))


### MF Model

In [14]:
import torch
import torch.nn as nn


LATENT_DIM = 16


class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=LATENT_DIM):
        super().__init__()

        self.user_embeddings = nn.Embedding(
            num_users,
            latent_dim,
            sparse=True,
        )

        self.item_embeddings = nn.Embedding(
            num_items,
            latent_dim,
            sparse=True,
        )

        self.user_bias = nn.Embedding(
            num_users,
            1,
            sparse=True,
        )

        self.item_bias = nn.Embedding(
            num_items,
            1,
            sparse=True,
        )

        self.global_bias = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embeddings(user_idx)
        item_vec = self.item_embeddings(item_idx)

        interaction = (user_vec * item_vec).sum(dim=1)

        user_bias = self.user_bias(user_idx).squeeze(-1)
        item_bias = self.item_bias(item_idx).squeeze(-1)

        return (
            interaction
            + user_bias
            + item_bias
            + self.global_bias
        )

In [15]:
num_users = len(users)
num_items = len(mf_items)

print("Users:", num_users)
print("Items:", num_items)
print("Latent dimensions:", LATENT_DIM)

Users: 16058864
Items: 1395130
Latent dimensions: 16


In [16]:
model = MatrixFactorization(
    num_users=num_users,
    num_items=num_items,
    latent_dim=LATENT_DIM,
)

print(model)

MatrixFactorization(
  (user_embeddings): Embedding(16058864, 16, sparse=True)
  (item_embeddings): Embedding(1395130, 16, sparse=True)
  (user_bias): Embedding(16058864, 1, sparse=True)
  (item_bias): Embedding(1395130, 1, sparse=True)
)


In [18]:
sum(p.numel() for p in model.parameters())

296717899

In [19]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)

    print("GPU:", props.name)
    print("Total GPU memory:",
          round(props.total_memory / 1024**3, 2), "GB")

    print("Currently allocated:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

    print("Currently reserved:",
          round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Total GPU memory: 3.68 GB
Currently allocated: 0.0 GB
Currently reserved: 0.0 GB


### CPU training test

In [20]:
device = torch.device("cpu")
model = model.to(device)

print("Model device:", model.user_embeddings.weight.device)

Model device: cpu


In [21]:
batch_size = 1024

test_users = torch.randint(
    0,
    num_users,
    (batch_size,),
    dtype=torch.long,
)

test_items = torch.randint(
    0,
    num_items,
    (batch_size,),
    dtype=torch.long,
)

test_targets = torch.randn(batch_size)

In [22]:
predictions = model(test_users, test_items)

loss = nn.functional.mse_loss(
    predictions,
    test_targets,
)

loss.backward()

print("Prediction shape:", predictions.shape)
print("Loss:", loss.item())
print("Forward/backward test: successful")

Prediction shape: torch.Size([1024])
Loss: 18.205202102661133
Forward/backward test: successful


### Sanity check for CPU training complete

In [17]:
sparse_parameters = [
    model.user_embeddings.weight,
    model.item_embeddings.weight,
    model.user_bias.weight,
    model.item_bias.weight,
]

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
)

In [18]:
print(type(optimizer).__name__)

SGD


### Training one real batch

In [20]:
import duckdb

TRAINING_PATH = (
    PROJECT_ROOT / "data" / "processed" / "mf_training" / "*.parquet"
)

con = duckdb.connect()

batch_df = con.execute(f"""
    SELECT
        user_idx,
        mf_item_idx,
        target
    FROM read_parquet('{TRAINING_PATH}')
    LIMIT 1024
""").fetchdf()

con.close()

print(batch_df.shape)
print(batch_df.head())

(1024, 3)
   user_idx  mf_item_idx  target
0  15728703       114907     0.0
1  15728740       665371     1.0
2  15728842      1394026     1.0
3  15729049       812987    -1.0
4  15729074       637996    -1.0


In [21]:
batch_users = torch.tensor(
    batch_df["user_idx"].to_numpy(),
    dtype=torch.long,
)

batch_items = torch.tensor(
    batch_df["mf_item_idx"].to_numpy(),
    dtype=torch.long,
)

batch_targets = torch.tensor(
    batch_df["target"].to_numpy(),
    dtype=torch.float32,
)

In [22]:
print("Users:", batch_users.shape)
print("Items:", batch_items.shape)
print("Targets:", batch_targets.shape)

print("User index range:",
      batch_users.min().item(),
      batch_users.max().item())

print("Item index range:",
      batch_items.min().item(),
      batch_items.max().item())

Users: torch.Size([1024])
Items: torch.Size([1024])
Targets: torch.Size([1024])
User index range: 15728703 15804303
Item index range: 232 1394220


In [23]:
model.train()

optimizer.zero_grad()

predictions = model(
    batch_users,
    batch_items,
)

loss = nn.functional.mse_loss(
    predictions,
    batch_targets,
)

loss.backward()

optimizer.step()

print("Training step successful")
print("Loss:", loss.item())

Training step successful
Loss: 18.34758186340332


### MF optimization

- Tested SparseAdam with the 16-dimensional MF architecture.
- The first real optimization step caused a kernel crash under the available
  14 GB system RAM, indicating excessive optimizer-state memory.
- Switched to SGD, which successfully completed a real sparse-embedding
  optimization step.
- Chose SGD for the first baseline to minimize optimizer memory and establish
  a reliable training configuration before considering more sophisticated
  optimizers.